# Exp 07: Best Practice

目标：把前面实验整理成一个可复用的 `torch.compile` 训练模板。

四条核心法则：
1. 显式 `mark_dynamic`，让动态维度第一次调用就进入 dynamic graph。
2. DataLoader 侧 `drop_last=True` 或保证 batch size `>= 2`，避开 `0/1 specialization`。
3. 多卡训练时先包 DDP，再 `torch.compile(ddp_model)`。
4. 诊断从 `TORCH_LOGS=recompiles` 和 `counters` 开始，必要时用 `tlparse` 做离线分析。

本 notebook 用一个小型 Transformer Encoder 模拟真实训练路径：动态 batch、BF16 autocast、optimizer step、data-independent 分支和调试断言。

In [ ]:
import random
import statistics
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch._dynamo.utils import counters

assert torch.cuda.is_available(), "This notebook requires a CUDA GPU."

DEVICE = "cuda"
DTYPE = torch.bfloat16
D_MODEL = 512
N_HEADS = 8
D_FF = 2048
DROPOUT = 0.0
SEQ_LEN = 128
MAX_BATCH = 64
MIN_BATCH = 4
WARMUP = 5
TRAIN_STEPS = 40

BATCH_SCHEDULE = ([32] * 10 + [16] * 6 + [48] * 6 + [32] * 8 + [24] * 6 + [64] * 4)[:TRAIN_STEPS]

torch.manual_seed(42)
random.seed(42)

## 模型

这里的分支都是 data-independent：

- `torch.compiler.is_compiling()` 保护 eager-only 断言。
- `self.training` 在编译时可被 guard，因此不会因为每个 step 的 tensor 值变化而 graph break。

注意不要在 forward 中写 `if x.sum().item() > ...` 这类 data-dependent Python 分支。

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff1 = nn.Linear(d_model, d_ff)
        self.ff2 = nn.Linear(d_ff, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor | None = None) -> torch.Tensor:
        if not torch.compiler.is_compiling():
            assert x.dim() == 3, f"expected (B, T, D), got {tuple(x.shape)}"

        normed = self.norm1(x)
        attn_out, _ = self.attn(
            normed,
            normed,
            normed,
            attn_mask=attn_mask,
            need_weights=False,
        )
        x = x + self.drop(attn_out)

        normed = self.norm2(x)
        hidden = F.gelu(self.ff1(normed))
        if self.training:
            hidden = self.drop(hidden)
        return x + self.ff2(hidden)


class SmallTransformer(nn.Module):
    def __init__(
        self,
        d_model: int = D_MODEL,
        n_heads: int = N_HEADS,
        d_ff: int = D_FF,
        n_layers: int = 2,
        vocab_size: int = 256,
        n_classes: int = 10,
    ) -> None:
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList(
            TransformerBlock(d_model, n_heads, d_ff, DROPOUT) for _ in range(n_layers)
        )
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        x = self.embed(token_ids)
        for layer in self.layers:
            x = layer(x)
        cls = self.norm(x[:, 0, :])
        return self.head(cls)

## 编译与训练工具

当前 PyTorch 版本不允许 `mode` 和 `options` 同时传给 `torch.compile`。因此这里为了展示 `shape_padding` / `epilogue_fusion`，只传 `options`，不再额外传 `mode="default"`。

`fused=True` 的 AdamW 是 CUDA 专用优化，能减少 optimizer 里的 kernel launch 开销；如果环境不支持，代码会回退到普通 AdamW。

In [ ]:
def configure_dynamo(ci_mode: bool = False) -> None:
    torch._dynamo.config.recompile_limit = 16
    torch._dynamo.config.accumulated_recompile_limit = 256

    if ci_mode:
        torch._dynamo.config.recompile_limit = 2
        torch._dynamo.config.fail_on_recompile_limit_hit = True


def build_compiled_model() -> nn.Module:
    model = SmallTransformer().to(DEVICE)
    return torch.compile(
        model,
        fullgraph=False,
        dynamic=None,
        options={"shape_padding": True, "epilogue_fusion": True},
    )


def build_optimizer(model: nn.Module) -> torch.optim.Optimizer:
    try:
        return torch.optim.AdamW(model.parameters(), lr=1e-4, fused=True)
    except RuntimeError:
        return torch.optim.AdamW(model.parameters(), lr=1e-4)


def make_batch(batch_size: int) -> tuple[torch.Tensor, torch.Tensor]:
    token_ids = torch.randint(0, 256, (batch_size, SEQ_LEN), device=DEVICE)
    labels = torch.randint(0, 10, (batch_size,), device=DEVICE)
    return token_ids, labels


def mark_batch_dynamic(token_ids: torch.Tensor, labels: torch.Tensor) -> None:
    torch._dynamo.mark_dynamic(token_ids, 0, min=MIN_BATCH, max=MAX_BATCH)
    torch._dynamo.mark_dynamic(labels, 0, min=MIN_BATCH, max=MAX_BATCH)


def train_step(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    token_ids: torch.Tensor,
    labels: torch.Tensor,
) -> float:
    mark_batch_dynamic(token_ids, labels)

    with torch.autocast("cuda", dtype=DTYPE):
        logits = model(token_ids)
        loss = F.cross_entropy(logits, labels)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return float(loss.detach())

## Eager Baseline

先跑未编译版本，后面用 p50/p95 step time 对比 compiled 稳态。这里包含 forward、loss、backward、optimizer step。

In [ ]:
def timed_step(fn) -> float:
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize()
    start.record()
    fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end)


def percentile(values: list[float], pct: float) -> float:
    values = sorted(values)
    index = int(len(values) * pct / 100)
    return values[min(index, len(values) - 1)]


def run_eager_baseline() -> list[float]:
    model = SmallTransformer().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    step_times = []

    for batch_size in BATCH_SCHEDULE:
        token_ids, labels = make_batch(batch_size)

        def step() -> None:
            with torch.autocast("cuda", dtype=DTYPE):
                logits = model(token_ids)
                loss = F.cross_entropy(logits, labels)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        step_times.append(timed_step(step))

    return step_times


eager_times = run_eager_baseline()
eager_p50 = percentile(eager_times, 50)
eager_p95 = percentile(eager_times, 95)
print(f"eager p50={eager_p50:.1f} ms  p95={eager_p95:.1f} ms")

## Compiled Training

warmup 阶段会触发编译，所以不要把前几步直接拿来和 eager 稳态比较。这里记录所有 step 时间，最后去掉 warmup 后再看 p50/p95。

In [ ]:
def run_compiled_training(ci_mode: bool = False) -> tuple[list[float], int]:
    configure_dynamo(ci_mode)
    model = build_compiled_model()
    optimizer = build_optimizer(model)
    counters.clear()
    step_times = []

    for step, batch_size in enumerate(BATCH_SCHEDULE):
        token_ids, labels = make_batch(batch_size)

        def step_fn() -> None:
            train_step(model, optimizer, token_ids, labels)

        elapsed_ms = timed_step(step_fn)
        step_times.append(elapsed_ms)

        if step < WARMUP or step % 10 == 0:
            print(
                f"step={step:2d} batch={batch_size:2d} "
                f"time={elapsed_ms:8.1f} ms "
                f"unique_graphs={counters['stats']['unique_graphs']}"
            )

    return step_times, counters["stats"]["unique_graphs"]


compiled_times, final_unique_graphs = run_compiled_training(ci_mode=False)
steady_times = compiled_times[WARMUP + 2 :]
compiled_p50 = percentile(steady_times, 50)
compiled_p95 = percentile(steady_times, 95)

print(f"compiled steady p50={compiled_p50:.1f} ms  p95={compiled_p95:.1f} ms")
print(f"final unique_graphs={final_unique_graphs}")

In [ ]:
print(f"{'mode':20s} {'p50(ms)':>10s} {'p95(ms)':>10s} {'speedup':>10s}")
print("-" * 56)
print(f"{'eager':20s} {eager_p50:10.1f} {eager_p95:10.1f} {'1.00x':>10s}")
print(
    f"{'compiled steady':20s} {compiled_p50:10.1f} {compiled_p95:10.1f} "
    f"{eager_p50 / compiled_p50:9.2f}x"
)

if final_unique_graphs <= 2:
    print(f"unique_graphs={final_unique_graphs}: recompile 控制良好。")
else:
    print(f"unique_graphs={final_unique_graphs}: 仍有 recompile，建议运行 TORCH_LOGS=recompiles。")

## DDP 说明

多卡训练时的顺序通常是：

```python
ddp_model = DistributedDataParallel(
    model,
    device_ids=[rank],
    static_graph=True,
    find_unused_parameters=False,
)
compiled = torch.compile(ddp_model, ...)
```

核心原则是：**DDP 在内，compile 在外**。这样 Dynamo 能看到 DDP wrapper，并让 DDPOptimizer 正常识别 backward bucket 边界，从而保留 backward compute 与 all-reduce overlap 的机会。

## 小结

综合最佳实践：

1. 对动态 batch / sequence 维显式 `mark_dynamic`，并设置合理 `min/max`。
2. 训练数据侧避免 batch size 为 0 或 1，优先 `drop_last=True`。
3. `mode` 和 `options` 在部分 PyTorch 版本中不能同时指定；需要自定义 Inductor options 时，只传 `options`。
4. forward 中保留 shape/dtype/config 这类 data-independent 分支，避免 tensor value 参与 Python `if`。
5. 诊断顺序固定化：先看 `unique_graphs`，再开 `TORCH_LOGS=recompiles`，最后用 `tlparse` 做全局分析。

确认稳定后，再尝试 `max-autotune`、bucketing、FSDP/DDP、compiled optimizer 等更激进方案。